# P6: NPDES Facility Merge

Primary merge per `MERGE.md` (§2, **P6**). Rolls up the NPDES point-source
compliance record into one wide table at **1 row per `npdes_id` + `fiscal_year`**,
carrying `wbd_huc12` / `wbd_huc8` so the downstream **S2** merge can attach
point-source pressure to a station via its watershed.

**Inputs** (all `data/tabular/02_clean/npdes/...`):
- `npdes-dmrs-clean.csv` (2,786,182 rows, one row per permit / outfall /
  parameter / period) — **aggregated** to `npdes_id` + `fiscal_year`. This is the
  only fiscal-year-varying input and therefore defines the output row set.
- `npdes-catchments-clean.csv` — NHDPlus catchment / HUC-12 linkage per permit.
- `echo-facilities-clean.csv` — facility identity, location, county, §303(d) flag.
- `echo-naics-clean.csv` / `echo-sics-clean.csv` — industry classification.
- `npdes-attains-clean.csv` — Clean Water Act assessment-unit condition per permit.

**Output:** `data/03a_merge_primary/npdes-facility.csv`, one row per permit +
fiscal year.

**Design notes**
- **DMRs are the base (left) table.** Fiscal year exists only in the DMR feed, so
  the grain `npdes_id` + `fiscal_year` (14,030 permit-years, 1,469 permits) comes
  from it; every other input is a *static-per-permit* table broadcast onto it by
  `npdes_id`. Permits that filed no DMR (≈750 of the 2,216 ECHO facilities) have
  no fiscal year and so do not appear — that is a property of the grain, not a
  join defect.
- **DMR aggregation avoids the per-outfall/per-parameter fan-out** (`MERGE.md`
  §6). Each permit-year is reduced to compliance/activity counts that are
  meaningful across mixed parameters and units — record/outfall/parameter counts,
  exceedance and violation counts, and late-report counts, plus max/mean of the
  unit-independent `exceedence_pct` and `days_late`. A cross-parameter mean of
  `dmr_value` is deliberately **not** computed: it would average incommensurable
  units (mg/L, pH, MGD, …) into a meaningless number.
- **Each static table is reduced to one row per `npdes_id` before joining**, so
  no join can fan the base out:
  - *Catchments* — a permit can list several sub-facility catchments (112 do),
    but 20 of those span more than one HUC-12. The first catchment per permit
    (ordered by `sub_id`) is taken as representative and the count retained as
    `npdes_n_catchments`.
  - *NAICS / SIC* — reduced to the facility's **primary** code (`is_primary`).
  - *ATTAINS* — a permit can link to several assessment units (73 do); reduced to
    counts (`attains_n_assessment_units`, `attains_n_impaired`) and an
    `attains_any_impaired` flag rather than one arbitrary unit's condition.

In [1]:
import os

import pandas as pd

CLEAN = "../../data/tabular/02_clean"
OUT_DIR = "../../data/03a_merge_primary"
OUT_FILE = f"{OUT_DIR}/npdes-facility.csv"

KEY = ["npdes_id", "fiscal_year"]

## Step 1: DMRs → per-permit-year compliance aggregates (base table)

The 2.79M-row DMR feed is read with only the columns the aggregates need
(`perm_feature_nmbr` is read as string — it mixes numeric and zero-padded outfall
labels). The reduction to `npdes_id` + `fiscal_year` produces:

- **activity counts** — `dmr_n_records`, `dmr_n_outfalls` (distinct outfalls),
  `dmr_n_parameters` (distinct regulated parameters);
- **exceedances** — `dmr_n_exceedances` (rows with `exceedence_pct > 0`) plus the
  `mean`/`max` of `exceedence_pct` over rows where a limit applied;
- **violations** — `dmr_n_violations` (rows carrying a `violation_code`);
- **reporting timeliness** — `dmr_n_late_reports` (rows with `days_late > 0`) and
  the `mean`/`max` of `days_late` over those late rows.

`(npdes_id, fiscal_year)` is asserted unique on the result — it is the output
grain.

In [2]:
DMR_COLS = ["npdes_id", "fiscal_year", "perm_feature_nmbr", "parameter_code",
            "exceedence_pct", "violation_code", "days_late"]
dmr = pd.read_csv(f"{CLEAN}/npdes/npdes-dmrs-clean.csv", usecols=DMR_COLS,
                  dtype={"perm_feature_nmbr": str}, low_memory=False)
dmr["fiscal_year"] = dmr["fiscal_year"].astype(int)
print(f"DMR rows: {len(dmr):,} | permits: {dmr['npdes_id'].nunique():,} | "
      f"fiscal years {dmr['fiscal_year'].min()}-{dmr['fiscal_year'].max()}")

# Condition flags summed into counts (a limit only applies to a subset of rows).
dmr["_exceed"] = dmr["exceedence_pct"] > 0
dmr["_violation"] = dmr["violation_code"].notna()
dmr["_late"] = dmr["days_late"] > 0

grouped = dmr.groupby(KEY)
dmr_agg = grouped.agg(
    dmr_n_records=("parameter_code", "size"),
    dmr_n_outfalls=("perm_feature_nmbr", "nunique"),
    dmr_n_parameters=("parameter_code", "nunique"),
    dmr_n_exceedances=("_exceed", "sum"),
    dmr_mean_exceedence_pct=("exceedence_pct", "mean"),
    dmr_max_exceedence_pct=("exceedence_pct", "max"),
    dmr_n_violations=("_violation", "sum"),
    dmr_n_late_reports=("_late", "sum"),
    dmr_mean_days_late=("days_late", "mean"),
    dmr_max_days_late=("days_late", "max"),
).reset_index()

assert not dmr_agg.duplicated(subset=KEY).any(), "DMR aggregate not unique on (npdes_id, fiscal_year)"
print(f"DMR aggregate: {len(dmr_agg):,} permit-years x {dmr_agg.shape[1]} cols")
dmr_agg.head(3)

DMR rows: 2,786,182 | permits: 1,469 | fiscal years 2015-2025
DMR aggregate: 14,030 permit-years x 12 cols


,npdes_id,fiscal_year,dmr_n_records,dmr_n_outfalls,dmr_n_parameters,dmr_n_exceedances,dmr_mean_exceedence_pct,dmr_max_exceedence_pct,dmr_n_violations,dmr_n_late_reports,dmr_mean_days_late,dmr_max_days_late
0,IA0000035,2017,18,1,5,0,NaN,NaN,0,0,NaN,NaN
1,IA0000035,2018,18,1,5,0,NaN,NaN,0,0,NaN,NaN
2,IA0000035,2019,18,1,5,0,NaN,NaN,18,0,NaN,NaN


## Step 2: ECHO facility identity (1 row / permit)

`echo-facilities-clean.csv` is already unique on `npdes_id`. Kept: facility name,
type, city, county FIPS, coordinates, and the §303(d) impaired-waters flag. The
coordinates here are the canonical ECHO facility location (the catchment table
carries near-identical NAD83 coordinates, dropped in Step 3 to avoid duplication).

In [3]:
fac = pd.read_csv(f"{CLEAN}/npdes/echo-facilities-clean.csv", dtype=str)
assert not fac.duplicated(subset="npdes_id").any(), "facilities not unique on npdes_id"

fac = fac[["npdes_id", "facility_name", "facility_type_code", "city", "county_fips",
           "latitude", "longitude", "impaired_303d"]].rename(
    columns={"latitude": "facility_latitude", "longitude": "facility_longitude"})
print(f"facilities: {len(fac):,} permits")
fac.head(3)

facilities: 2,216 permits


,npdes_id,facility_name,facility_type_code,city,county_fips,facility_latitude,facility_longitude,impaired_303d
0,IA0000035,"OTTUMWA WATER WORKS, CITY OF",CTG,OTTUMWA,19179,41.01903,-92.41676,True
1,IA0000051,JOHN DEERE DUBUQUE WORKS,COR,DUBUQUE,19061,42.56591,-90.69351,True
2,IA0000060,JOHN DEERE WATERLOO WORKS (DRIVE TRAIN OPERATI...,COR,WATERLOO,19013,42.50304,-92.35298,True


## Step 3: NHDPlus catchment / HUC-12 linkage (1 row / permit)

Reduced to one representative catchment per permit — the first by `sub_id` — with
the raw catchment count retained. `wbd_huc8` is derived as the first 8 digits of
`wbd_huc12` for the downstream HUC-8 joins in S2. Read HUC/NHDPlus IDs as strings
to keep leading zeros.

In [4]:
cat = pd.read_csv(f"{CLEAN}/npdes/npdes-catchments-clean.csv",
                  dtype={"wbd_huc12": str, "nhdplusid": str, "sub_id": str})
cat_n = cat.groupby("npdes_id").size().rename("npdes_n_catchments")

cat1 = cat.sort_values(["npdes_id", "sub_id"]).drop_duplicates("npdes_id", keep="first").copy()
cat1["wbd_huc8"] = cat1["wbd_huc12"].str[:8]
assert cat1["wbd_huc12"].str.len().eq(12).all(), "non-12-digit wbd_huc12 present"

cat1 = cat1[["npdes_id", "wbd_huc12", "wbd_huc8", "wbd_huc12_name",
             "nhdplusid", "area_sqkm"]].merge(cat_n, on="npdes_id")
assert not cat1.duplicated(subset="npdes_id").any(), "catchment reduction not unique on npdes_id"
print(f"catchments: {len(cat1):,} permits | permits with >1 catchment: {(cat_n > 1).sum()}")
cat1.head(3)

catchments: 2,156 permits | permits with >1 catchment: 112


,npdes_id,wbd_huc12,wbd_huc8,wbd_huc12_name,nhdplusid,area_sqkm,npdes_n_catchments
0,IA0000035,071000090709,07100009,Kettle Creek-Des Moines River,4995479,6.2667,1
1,IA0000051,070600030604,07060003,Lower Little Maquoketa River,13325396,0.8550,6
2,IA0000060,070802050906,07080205,Sink Creek-Cedar River,22472185,0.2340,2


## Step 4: Industry classification — primary NAICS & SIC (1 row / permit)

Both ECHO industry tables are long (one row per permit + code). Each is reduced to
the permit's **primary** classification by sorting `is_primary` first and keeping
the top row per permit.

In [5]:
def primary_industry(filename, prefix):
    d = pd.read_csv(f"{CLEAN}/npdes/{filename}", dtype=str)
    d["is_primary"] = d["is_primary"].map({"True": True, "False": False})
    d = d.sort_values(["npdes_id", "is_primary"], ascending=[True, False])
    d = d.drop_duplicates("npdes_id", keep="first")
    out = d[["npdes_id", "code", "description"]].rename(
        columns={"code": f"{prefix}_code", "description": f"{prefix}_desc"})
    assert not out.duplicated(subset="npdes_id").any(), f"{prefix} reduction not unique on npdes_id"
    return out

naics = primary_industry("echo-naics-clean.csv", "naics")
sics = primary_industry("echo-sics-clean.csv", "sic")
print(f"NAICS: {len(naics):,} permits | SIC: {len(sics):,} permits")
naics.head(3)

NAICS: 1,668 permits | SIC: 1,692 permits


,npdes_id,naics_code,naics_desc
0,IA0000035,221310,Water Supply and Irrigation Systems
1,IA0000051,333111,Farm Machinery and Equipment Manufacturing
2,IA0000060,333111,Farm Machinery and Equipment Manufacturing


## Step 5: ATTAINS assessment condition (1 row / permit)

A permit can link to several assessed water bodies, so ATTAINS is reduced to
per-permit counts rather than one arbitrary unit's condition. `water_condition`
values beginning `Impaired` (any 303(d)/restoration variant) count as impaired.

In [6]:
att = pd.read_csv(f"{CLEAN}/npdes/npdes-attains-clean.csv", dtype=str)
att["_impaired"] = att["water_condition"].fillna("").str.startswith("Impaired")

att_agg = att.groupby("npdes_id").agg(
    attains_n_assessment_units=("assessment_unit_id", "nunique"),
    attains_n_impaired=("_impaired", "sum"),
).reset_index()
att_agg["attains_any_impaired"] = att_agg["attains_n_impaired"] > 0
assert not att_agg.duplicated(subset="npdes_id").any(), "ATTAINS reduction not unique on npdes_id"
print(f"ATTAINS: {len(att_agg):,} permits | with any impaired AU: {att_agg['attains_any_impaired'].sum():,}")
att_agg.head(3)

ATTAINS: 1,018 permits | with any impaired AU: 551


,npdes_id,attains_n_assessment_units,attains_n_impaired,attains_any_impaired
0,COPIU0087,1,1,True
1,IA0000035,1,1,True
2,IA0000051,2,1,True


## Step 6: Assemble and save

Each static table is unique on `npdes_id`, so left-joining them onto the DMR base
can neither add nor drop rows — asserted by an unchanged row count and a re-check
of the `(npdes_id, fiscal_year)` grain. Columns are ordered keys → watershed
linkage → facility identity → industry → ATTAINS → DMR aggregates.

In [7]:
df = (dmr_agg
      .merge(fac, on="npdes_id", how="left")
      .merge(cat1, on="npdes_id", how="left")
      .merge(naics, on="npdes_id", how="left")
      .merge(sics, on="npdes_id", how="left")
      .merge(att_agg, on="npdes_id", how="left"))

assert len(df) == len(dmr_agg), f"row count changed on join: {len(df)} vs {len(dmr_agg)}"
assert not df.duplicated(subset=KEY).any(), "output grain violated: duplicate (npdes_id, fiscal_year)"

spatial_cols = ["wbd_huc12", "wbd_huc8", "wbd_huc12_name", "nhdplusid", "area_sqkm", "npdes_n_catchments"]
fac_cols = ["facility_name", "facility_type_code", "city", "county_fips",
            "facility_latitude", "facility_longitude", "impaired_303d"]
ind_cols = ["naics_code", "naics_desc", "sic_code", "sic_desc"]
att_cols = ["attains_n_assessment_units", "attains_n_impaired", "attains_any_impaired"]
dmr_cols = [c for c in dmr_agg.columns if c.startswith("dmr_")]

df = df[KEY + spatial_cols + fac_cols + ind_cols + att_cols + dmr_cols]
df = df.sort_values(KEY).reset_index(drop=True)

print(f"Final shape: {df.shape}")
print(f"permit-years: {len(df):,} | distinct permits: {df['npdes_id'].nunique():,} | "
      f"fiscal years {df['fiscal_year'].min()}-{df['fiscal_year'].max()}")
print(f"permit-years with a HUC-12 linkage: {df['wbd_huc12'].notna().sum():,} / {len(df):,}")
print(f"permit-years matched to an ECHO facility: {df['facility_name'].notna().sum():,} / {len(df):,}")
df.head(3)

Final shape: (14030, 32)
permit-years: 14,030 | distinct permits: 1,469 | fiscal years 2015-2025
permit-years with a HUC-12 linkage: 13,589 / 14,030
permit-years matched to an ECHO facility: 14,030 / 14,030


,npdes_id,fiscal_year,wbd_huc12,wbd_huc8,wbd_huc12_name,nhdplusid,area_sqkm,npdes_n_catchments,facility_name,facility_type_code,...,dmr_n_records,dmr_n_outfalls,dmr_n_parameters,dmr_n_exceedances,dmr_mean_exceedence_pct,dmr_max_exceedence_pct,dmr_n_violations,dmr_n_late_reports,dmr_mean_days_late,dmr_max_days_late
0,IA0000035,2017,071000090709,07100009,Kettle Creek-Des Moines River,4995479,6.2667,1.0,"OTTUMWA WATER WORKS, CITY OF",CTG,...,18,1,5,0,NaN,NaN,0,0,NaN,NaN
1,IA0000035,2018,071000090709,07100009,Kettle Creek-Des Moines River,4995479,6.2667,1.0,"OTTUMWA WATER WORKS, CITY OF",CTG,...,18,1,5,0,NaN,NaN,0,0,NaN,NaN
2,IA0000035,2019,071000090709,07100009,Kettle Creek-Des Moines River,4995479,6.2667,1.0,"OTTUMWA WATER WORKS, CITY OF",CTG,...,18,1,5,0,NaN,NaN,18,0,NaN,NaN


In [8]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 14,030 rows x 32 cols -> ../../data/03a_merge_primary/npdes-facility.csv
